In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current device: {device}")

Current device: cuda


# 1. Utility Functions

In [2]:
import matplotlib.pyplot as plt
from IPython import display
class Animator:
    """在 Jupyter 训练循环中实时刷新曲线。分类任务用双子图：上 loss、下 acc。"""

    def __init__(self, xlabel='epoch', xlim=None,
                 loss_legend=None, acc_legend=None, acc_ylim=(0, 1),
                 fmts=('-', 'm--', 'g-.', 'r:', 'c-', 'y--', 'k-.', 'b:'),
                 figsize=(5, 6)):
        self.xlabel = xlabel
        self.xlim = xlim
        self.loss_legend = list(loss_legend) if loss_legend else []
        self.acc_legend = list(acc_legend) if acc_legend else []
        self.acc_ylim = acc_ylim
        self.fmts = list(fmts)
        self.loss_X, self.loss_Y = None, None
        self.acc_X, self.acc_Y = None, None

        if self.acc_legend:
            self.fig, (self.ax_loss, self.ax_acc) = plt.subplots(
                2, 1, figsize=figsize, sharex=True)
        else:
            self.fig, self.ax_loss = plt.subplots(1, 1, figsize=(figsize[0], figsize[1] / 2))
            self.ax_acc = None

    @staticmethod
    def _as_lines(values):
        if values is None:
            return []
        if isinstance(values, (list, tuple)):
            return list(values)
        return [values]

    def _ensure_lines(self, storage, n):
        X, Y = storage
        if X is None:
            return [[] for _ in range(n)], [[] for _ in range(n)]
        if n > len(X):
            X.extend([] for _ in range(n - len(X)))
            Y.extend([] for _ in range(n - len(Y)))
        return X, Y

    def _append_points(self, storage, x, values):
        values = self._as_lines(values)
        if not values:
            return storage
        x = self._as_lines(x)
        if len(x) == 1:
            x = x * len(values)
        elif len(x) != len(values):
            raise ValueError(
                f'x 与 values 长度不一致: len(x)={len(x)}, len(values)={len(values)}'
            )
        X, Y = self._ensure_lines(storage, len(values))
        for i, (xi, yi) in enumerate(zip(x, values)):
            if xi is not None and yi is not None:
                X[i].append(xi)
                Y[i].append(yi)
        return X, Y

    def _plot_panel(self, ax, X, Y, legend, ylim=None, show_xlabel=False):
        ax.cla()
        if X is None:
            return
        for i, (xs, ys) in enumerate(zip(X, Y)):
            fmt = self.fmts[i % len(self.fmts)]
            label = legend[i] if i < len(legend) else None
            ax.plot(xs, ys, fmt, label=label)
        if self.xlim:
            ax.set_xlim(self.xlim)
        if ylim:
            ax.set_ylim(ylim)
        if show_xlabel and self.xlabel:
            ax.set_xlabel(self.xlabel)
        if legend:
            ax.legend(legend)
        ax.grid(True)

    def add(self, x, loss=None, acc=None):
        """追加数据并重绘。分类: add(epoch, loss=(tr, te), acc=(tr, te))。"""
        self.loss_X, self.loss_Y = self._append_points(
            (self.loss_X, self.loss_Y), x, loss)
        if self.ax_acc is not None:
            self.acc_X, self.acc_Y = self._append_points(
                (self.acc_X, self.acc_Y), x, acc)

        self._plot_panel(self.ax_loss, self.loss_X, self.loss_Y,
                         self.loss_legend, show_xlabel=self.ax_acc is None)
        if self.ax_acc is not None:
            self._plot_panel(self.ax_acc, self.acc_X, self.acc_Y,
                             self.acc_legend, ylim=self.acc_ylim, show_xlabel=True)

        self.fig.tight_layout()
        display.clear_output(wait=True)
        display.display(self.fig)

    def close(self):
        plt.close(self.fig)

In [3]:
import os
import re
import hashlib
from urllib import request

DATA_URL = 'http://d2l-data.s3-accelerate.amazonaws.com/'
DATA_HUB = {
    'time_machine': (DATA_URL + 'timemachine.txt',
                     '090b5e7e70c295757f55df93cb0a180b9691891a')
}

def download(name, cache_dir='./data'):
    """下载 DATA_HUB 中的文件，返回本地路径。"""
    assert name in DATA_HUB, f"{name} 不存在于 {DATA_HUB}"
    url, sha1_hash = DATA_HUB[name]
    os.makedirs(cache_dir, exist_ok=True)
    fname = os.path.join(cache_dir, url.split('/')[-1])
    if os.path.exists(fname):
        sha1 = hashlib.sha1()
        with open(fname, 'rb') as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname
    print(f'正在从 {url} 下载 {fname}...')
    request.urlretrieve(url, fname)
    return fname

def read_time_machine():
    """读取 H.G. Wells《时光机器》，返回预处理后的文本行列表。"""
    with open(download('time_machine'), 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

In [4]:
def tokenize(lines, token='word'):
    """将文本行拆分为单词或字符词元。"""
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print(f"错误: 未知词元类型: '{token}'")

In [ ]:
import collections

def count_corpus(tokens):
    if isinstance(tokens, list):
        tokens = [tk for line in tokens for tk in line]
    return collections.Counter(tokens)

In [ ]:
class Vocab:
    """文本词表"""
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        # 按出现频率排序
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1],
                                   reverse=True)
        # 未知词元的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx
                             for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):  # 未知词元的索引为0
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

# 2. Data reloading and preprocessing.

In [7]:
lines = read_time_machine()
tokens_chars = tokenize(lines, 'char')
tokens_words = tokenize(lines, 'word')
for i in range(5):
    print(lines[i*3])
    print(tokens_chars[i*3])
    print(tokens_words[i*3])

the time machine by h g wells
['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm', 'a', 'c', 'h', 'i', 'n', 'e', ' ', 'b', 'y', ' ', 'h', ' ', 'g', ' ', 'w', 'e', 'l', 'l', 's']
['the', 'time', 'machine', 'by', 'h', 'g', 'wells']

[]
[]

[]
[]
was expounding a recondite matter to us his grey eyes shone and
['w', 'a', 's', ' ', 'e', 'x', 'p', 'o', 'u', 'n', 'd', 'i', 'n', 'g', ' ', 'a', ' ', 'r', 'e', 'c', 'o', 'n', 'd', 'i', 't', 'e', ' ', 'm', 'a', 't', 't', 'e', 'r', ' ', 't', 'o', ' ', 'u', 's', ' ', 'h', 'i', 's', ' ', 'g', 'r', 'e', 'y', ' ', 'e', 'y', 'e', 's', ' ', 's', 'h', 'o', 'n', 'e', ' ', 'a', 'n', 'd']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
lights in the lilies of silver caught the bubbles that flashed and
['l', 'i', 'g', 'h', 't', 's', ' ', 'i', 'n', ' ', 't', 'h', 'e', ' ', 'l', 'i', 'l', 'i', 'e', 's', ' ', 'o', 'f', ' ', 's', 'i', 'l', 'v', 'e', 'r', ' ', 'c', 'a', 'u', 'g', 'h', 't', ' ', 't', 'h', 'e', ' ',

In [ ]:
vocab = Vocab(tokens_words)